# Architecture

## Tokenisation + Data Loading

In [1]:
import os

from socrates_ai.data_prep import TheFarmer
from socrates_ai.oracle import Translator

farmer = TheFarmer()
tokeniser_path = os.path.join('..', 'data', 'tokenizer.json')

translator = Translator(cleaned_books_path = farmer.clean_dir)
oracle_tokens = translator.load_tokeniser(path=tokeniser_path)

In [2]:
import glob
import random

import torch


def load_book_paths(clean_dir):
    # sorted() so the file order is deterministic across runs -- without it,
    # glob's order isn't guaranteed, which would make the train/val split
    # below silently different every time we re-run this notebook
    return sorted(glob.glob(os.path.join(clean_dir, "*.txt")))


def tokenise_book(path, tokeniser):
    with open(path, encoding="utf-8") as f:
        text = f.read()
    ids = tokeniser.encode(text).ids

    bos_id = tokeniser.token_to_id("<bos>")
    eos_id = tokeniser.token_to_id("<eos>")
    # wrap each book in <bos>/<eos> before concatenating books together later --
    # without this the model would see one book's ending flow straight into the
    # next book's opening with no signal that anything changed
    return [bos_id] + ids + [eos_id]

In [3]:
def split_books_train_val(book_paths, tokeniser, val_fraction=0.1, seed=231103):
    # fixed seed -> same split every time we re-run the notebook, so results
    # are comparable across experiments instead of validating against a
    # different random slice of books each run
    rng = random.Random(seed)
    shuffled = book_paths.copy()
    rng.shuffle(shuffled)

    n_val = max(1, int(len(shuffled) * val_fraction))
    val_paths = shuffled[:n_val]
    train_paths = shuffled[n_val:]

    # held-out WHOLE books, not a slice cut out of one long token stream --
    # this tests whether the model generalises to material it has never seen
    # any part of, rather than just memorising continuations of familiar text
    train_ids = [tid for path in train_paths for tid in tokenise_book(path, tokeniser)]
    val_ids = [tid for path in val_paths for tid in tokenise_book(path, tokeniser)]

    # long tensor because these are token ids (indices into the embedding
    # table), not continuous values
    return (
        torch.tensor(train_ids, dtype=torch.long),
        torch.tensor(val_ids, dtype=torch.long),
    )

In [4]:
def get_batch(data, block_size, batch_size, device="cpu"):
    # -1 so the last possible start index still leaves room for y to grab
    # one extra token past the end of x (y is x shifted forward by one
    # position -- next-token prediction needs that overhang)
    max_start = len(data) - block_size - 1
    start_idx = torch.randint(0, max_start, (batch_size,))

    # x: block_size tokens starting at i | y: the same window shifted by
    # one position -- at every position, y is "the token that actually
    # came next", which is what the model is trained to predict
    x = torch.stack([data[i : i + block_size] for i in start_idx])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in start_idx])

    return x.to(device), y.to(device)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

book_paths = load_book_paths(farmer.clean_dir)
print(f"{len(book_paths)} cleaned books found")

train_ids, val_ids = split_books_train_val(book_paths, oracle_tokens)
print(f"train tokens: {len(train_ids):,}")
print(f"val tokens:   {len(val_ids):,}")

x, y = get_batch(train_ids, block_size=128, batch_size=64, device=device)
print(f"\nsample batch -> x: {tuple(x.shape)}, y: {tuple(y.shape)}")

Using device: cpu
50 cleaned books found
train tokens: 5,735,147
val tokens:   439,408

sample batch -> x: (64, 128), y: (64, 128)


# Model

In [ ]:
### OLD MODEL FROM COURSEWORK

# Define your model here

class TextPredictionNetwork(nn.Module):
  """
  RNN language model (stacked GRU - 2 layers).

  Parameters
  ----------
      vocab_size: int, size of the user defined alphabet.

      emd_dim: int, (defualt 32) size of the character embedding
        dimension.

      hidden_dim: int, (defualt 128) size of the hidden layer dimensions
  """
  def __init__(self, vocab_size: int, emb_dim: int = 32, hidden_dim: int = 128):
    super().__init__()
    self.vocab_size = vocab_size
    self.emb_dim = emb_dim
    self.embed = nn.Embedding(vocab_size, emb_dim)
    self.hidden_dim = hidden_dim
    self.num_layers = 2

    self.dropout = nn.Dropout(p=0.05)   # Added in dropout for better generalisation
    self.rnn = nn.GRU(
        input_size=self.emb_dim,
        hidden_size=hidden_dim,
        num_layers = self.num_layers,
        batch_first=True,
        dropout=0.1
    )
    self.fc = nn.Linear(hidden_dim, vocab_size, bias=True)

  def forward(self, x: torch.Tensor, h: torch.Tensor | None = None):
    """
    forward pass
    """

    x_emb = self.embed(x)
    x_emb = self.dropout(x_emb)

    out, h = self.rnn(x_emb, h)
    out = self.dropout(out)
    logits = self.fc(out)

    return logits, h




  from sys import base_prefix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class NeuralOracle():
  """
  I've decided to create essentially a wrapper class for the
  functionality we want for this coursework, the idea is that it
  will help keep things clear and easy to debug. I also find it helps
  me layout the project, makes it easier to add/remove methods and helps
  breakdown the problem into smaller parts.

  Parameters
  ----------
  model: TextPredictionNetwork, an instance of the model class from above.
  raw_book: str, the raw text of the book. Use function load_text()
  train_split: float (default 0.9), the fraction of the book to use for training.

  """

  def __init__(self, model, raw_book, train_split=0.9):
    self.model = model.to(device)
    self.raw_book = raw_book

    # Hyperparameters
    self.train_split = train_split
    self.seq_len = 128
    self.batch_size = 64
    self.num_epochs = 8
    self.steps_per_epoch = 100
    self.eval_batches = 20
    self.lr = 1e-3

    self.history = []
    self.ids = encode(self.raw_book)
    split_idx = int(train_split * len(self.ids))
    self.train_ids = self.ids[:split_idx]
    self.val_ids   = self.ids[split_idx:]


  def set_runtime_params(self, **kwargs):
    """
    Sets the key runtime parameters for the model setup. Used if the user
    wants options different from the defualt.
    """
    valid_keys = {
        "train_split",
        "seq_len",
        "batch_size",
        "num_epochs",
        "steps_per_epoch",
        "eval_batches",
        "lr"
    }

    for key, val in kwargs.items():
      if key not in valid_keys:
        raise ValueError(f"Unknown hyperparameter: {key}")
      setattr(self, key, val)

  def print_model_setup(self):
    """
    very simple method to output the current state of the model,
    used for making notes when testing out different model configs.
    """
    print("\nHyperparameters:")
    print(f"  embedding dim   : {self.model.emb_dim}")
    print(f"  hidden dim      : {self.model.hidden_dim}")
    print(f"  train_split     : {self.train_split}")
    print(f"  seq_len         : {self.seq_len}")
    print(f"  batch_size      : {self.batch_size}")
    print(f"  num_epochs      : {self.num_epochs}")
    print(f"  steps_per_epoch : {self.steps_per_epoch}")
    print(f"  eval_batches    : {self.eval_batches}")
    print(f"  learning rate   : {self.lr}")

  def get_batch(self, data):
    """
    batch sampler method

    Parameters
    ----------
    data: dataset which requites batching, in our case this is the endocded
    raw book.

    Returns
    -------
    x, y: torch.Tensor, torch.Tensor

    """
    max_start = len(data) - self.seq_len - 1
    idx = torch.randint(0, max_start, (self.batch_size,))

    x = torch.stack([data[i:i+self.seq_len]     for i in idx])
    y = torch.stack([data[i+1:i+self.seq_len+1] for i in idx])

    return x.to(device), y.to(device)

  def evaluate(self):
    """
    evaluation method, this tracks key model
    performance metrics for the train/validation batches.

    Returns
    -------
    results: dict
    """
    self.model.eval()
    results = {}

    with torch.no_grad():
      for name, data in [("train", self.train_ids), ("val", self.val_ids)]:
        losses = []
        correct = 0
        correct_top3 = 0
        total = 0

        for _ in range(self.eval_batches):
          x, y = self.get_batch(data)
          logits, _ = self.model(x)

          B, T, V = logits.shape
          loss = F.cross_entropy(
              logits.view(B*T, V),
              y.view(B*T)
          )
          losses.append(loss.item())

          preds = logits.argmax(dim=-1)
          correct += (preds == y).sum().item()
          total   += y.numel()

          top3 = logits.topk(3, dim=-1).indices
          matches_top3 = (top3 == y.unsqueeze(-1))
          correct_top3 += matches_top3.any(dim=-1).sum().item()



        avg_loss = sum(losses)/len(losses)
        bpc = avg_loss / math.log(2)
        acc = correct/total
        top3_acc = correct_top3 / total

        results[name] = {
            "loss": avg_loss,
            "bpc": bpc,
            "acc": acc,
            "top3_acc": top3_acc,
        }

    self.model.train()
    return results

  def train_and_eval(self, return_results = False):
    """
    model training method, this is where the actual
    backpropagation happens

    Parameters
    ----------
    return_results: bool, this is only internally used for other methods in
    the class.
    """
    start_time = time.time()
    opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-2)
    for epoch in range(1, self.num_epochs + 1):
      if epoch == (self.num_epochs//2)+1:
        for g in opt.param_groups:
          g["lr"] *= 0.5
        print("Decayed lr to", opt.param_groups[0]["lr"])

      running_loss = 0.0

      for step in range(1, self.steps_per_epoch + 1):
        x, y = self.get_batch(self.train_ids)
        logits, _ = self.model(x)

        B, T, V = logits.shape
        loss = F.cross_entropy(
            logits.view(B*T, V),
            y.view(B*T)
        )

        opt.zero_grad()
        loss.backward()
        opt.step()

        running_loss += loss.item()

      metrics = self.evaluate()
      avg_train_loss = running_loss / self.steps_per_epoch
      avg_train_bpc = avg_train_loss / math.log(2)

      self.history.append({
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "train_bpc": avg_train_bpc,
        "val_loss": metrics["val"]["loss"],
        "val_bpc": metrics["val"]["bpc"],
        "val_acc": metrics["val"]["acc"],
        "val_top3_acc": metrics["val"]["top3_acc"],
      })

      if not return_results:
      # this is here so not every metric is printed during grid search
        print(f"\nEpoch {epoch}/{self.num_epochs}")
        print(f"  Train loss (epoch avg): {avg_train_loss:.4f}")
        for split in ["train", "val"]:
            m = metrics[split]
            print(
                f"  {split.capitalize():5s} | "
                f"loss: {m['loss']:.4f} | "
                f"bpc: {m['bpc']:.4f} | "
                f"acc: {m['acc']*100:6.2f}% | "
                f"top3 acc: {m["top3_acc"]*100:6.2f}%"
            )

    end_time = time.time()
    elapsed = end_time - start_time

    hrs = int(elapsed // 3600)
    mins = int((elapsed % 3600) // 60)
    secs = elapsed % 60

    if return_results:
      val_metrics = metrics["val"]
      return  val_metrics["loss"], val_metrics["bpc"], val_metrics["acc"], val_metrics["top3_acc"], elapsed

    print("\n==============================")
    print(f"Total training time: {hrs:02d}:{mins:02d}:{secs:05.2f}")
    print("==============================")


  def predict(self, prompt: str,
              length: int = 200, temperature: float = 1.0) -> str:
    """
    Generate text from the trained model, starting from `prompt`.

    Parameters
    ----------
    prompt: str, initial text to condition on.
    length: int, number of new characters to generate.
    temperature: >0 float, lower = more deterministic, higher = more random.

    Returns
    -------
    prompt + generated continuation.
    """
    assert temperature > 0, "Temperature must be > 0."

    self.model.eval()
    prompt_ids = encode(prompt).to(device)
    prompt_ids = prompt_ids.unsqueeze(0)

    generated_chars = [ch for ch in prompt]

    with torch.no_grad():
      logits, h = self.model(prompt_ids)
      last_id = prompt_ids[:, -1].view(1, 1)

      for _ in range(length):
        logits, h = self.model(last_id, h)
        logits_step = logits[0, -1] / temperature
        probs = F.softmax(logits_step, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)  # (1,)
        next_char = INV_VOCAB[int(next_id)]
        generated_chars.append(next_char)
        last_id = next_id.view(1, 1)

    self.model.train()
    return "".join(generated_chars)

  def plot_training_curves(self):
    """
    Plot train/validation loss and BPC over epochs using self.history.
    Call train_and_eval() once before using this.
    """
    if not hasattr(self, "history") or len(self.history) == 0:
      print("No training history found. Run train_and_eval() first.")
      return

    epochs     = [h["epoch"]       for h in self.history]
    train_loss = [h["train_loss"]  for h in self.history]
    val_loss   = [h["val_loss"]    for h in self.history]
    train_bpc  = [h["train_bpc"]   for h in self.history]
    val_bpc    = [h["val_bpc"]     for h in self.history]

    plt.figure(figsize=(10, 4))

    # Loss curves
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_loss, label="Train loss")
    plt.plot(epochs, val_loss,   label="Val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss over training")
    plt.legend()

    # BPC curves
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_bpc, label="Train BPC")
    plt.plot(epochs, val_bpc,   label="Val BPC")
    plt.xlabel("Epoch")
    plt.ylabel("Bits per character")
    plt.title("BPC over training")
    plt.legend()

    plt.tight_layout()
    plt.show()


  def talk(self, prompt: str, length: int = 200, temperature: float = 1.0) -> None:
    """
    Convenience method to print a sampled continuation. same parameters as
    self.predict()
    """
    print(f"\n--- Prompt: {repr(prompt)} | length={length} | T={temperature} ---")
    sample = self.predict(prompt=prompt, length=length, temperature=temperature)
    print(sample)


  def plot_next_char_distribution(self, prompt: str, temperature: float = 1.0):
    """
    function used to inspect the softmax distribution for the next character

    Parameters
    ----------
    prompt: str, initial text to condition on.
    temperature: >0 float, lower = more deterministic, higher = more random.
    """

    self.model.eval()
    with torch.no_grad():
      prompt_ids = encode(prompt).to(device).unsqueeze(0)  # (1, T)
      logits, h = self.model(prompt_ids)
      last_id = prompt_ids[:, -1].view(1, 1)

      logits, h = self.model(last_id, h)  # (1, 1, V)
      logits_step = logits[0, -1] / temperature
      probs = F.softmax(logits_step, dim=-1).cpu().numpy()

    chars = [INV_VOCAB[i] for i in range(len(probs))]

    plt.figure(figsize=(10, 4))
    plt.bar(range(len(probs)), probs)
    plt.xticks(range(len(probs)), chars, rotation=90)
    plt.title(f"Next-char distribution after prompt: {repr(prompt)} (T={temperature})")
    plt.ylabel("Probability")
    plt.tight_layout()
    plt.show()

  def plot_corpus_vs_model_char_freq(
    self,
    prompt: str = "the ",
    sample_length: int = 5000,
    temperature: float = 1.0,
    max_corpus_chars: int = 50000,
    ) -> None:
    """
    Compare character frequency in the training corpus vs
    a long sample generated by the model.
    """

    def char_freq(text: str) -> dict[str, float]:
      """
      helper function
      """
      counts = Counter(text)
      total = sum(counts.values())
      if total == 0:
        return {ch: 0.0 for ch in ALPHABET}
      return {ch: counts.get(ch, 0) / total for ch in ALPHABET}

    corpus_text = decode(self.train_ids[:max_corpus_chars])
    corpus_freq = char_freq(corpus_text)

    generated_text = self.predict(prompt=prompt, length=sample_length, temperature=temperature)
    model_freq = char_freq(generated_text)

    chars = ALPHABET
    corpus_vals = [corpus_freq.get(ch, 0.0) for ch in chars]
    model_vals  = [model_freq.get(ch, 0.0) for ch in chars]

    import numpy as np
    x = np.arange(len(chars))
    width = 0.4

    plt.figure(figsize=(12, 4))
    plt.bar(x - width/2, corpus_vals, width, label="Corpus")
    plt.bar(x + width/2, model_vals,  width, label="Model")

    plt.xticks(x, chars, rotation=90)
    plt.ylabel("Relative frequency")
    plt.title(f"Character frequency: corpus vs model (prompt={repr(prompt)}, T={temperature})")
    plt.legend()
    plt.tight_layout()
    plt.show()

  def grid_search(self, hidden_dims = [], embed_dims = [], lrs = [], **kwargs):
    """
    custom grid search method as a basic way of finding oout a good paramater
    combination

    Parameters
    ----------
    hidden_dims: list, list of hidden dimensions to try
    embed_dims: list, list of embedding dimensions to try
    lrs: list, list of learning rates to try
    **kwargs: used when the defualt model paramaters are NOT used, this is
    important, otherwise grid search results will be skewed.

    Returns
    -------
    df: pd.DataFrame, dataframe of all grid search results
    """

    all_combos = itertools.product(hidden_dims, embed_dims, lrs)
    results = []
    for hid_dim, emb_dim, lr in all_combos:
      print(f"\n=== Running model: hid={hid_dim}, emb={emb_dim}, lr={lr} ===")
      self.model = TextPredictionNetwork(vocab_size=L, emb_dim = emb_dim, hidden_dim=hid_dim).to(device)
      self.set_runtime_params(**kwargs, lr = lr)
      loss, bpc, acc, top3_acc, time = self.train_and_eval(return_results=True)
      results.append({
                  "hid_dim": hid_dim,
                  "emb_dim": emb_dim,
                  "lr": lr,
                  "loss": loss,
                  "bpc": bpc,
                  "acc": acc,
                  "top3_acc": top3_acc,
                  "time_s": time
              })
    return pd.DataFrame(results)


  def temperature_sweep(self, prompt="the ", length=200, temps=[0.25, 0.5, 0.75, 1, 1.25]):
    """
    Used for producing outputs for different temperatures based on the
    same prompt - useful for seeing the effect temperature has on the output.
    """
    print(f"Prompt: {repr(prompt)}")
    for T in temps:
      print(f"\n--- Temperature = {T} ---")
      print(self.predict(prompt=prompt, length=length, temperature=T))
